In [1]:
import argparse
import json
import os
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import scipy
import skimage
import tifffile
import tqdm
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.file_reading import (
    find_files_available,
    read_in_channels,
    read_zstack_image,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)
from image_analysis_3D.file_utils.read_in_channel_mapping import (
    retrieve_channel_mapping,
)
from matplotlib.colors import LinearSegmentedColormap

In [2]:
PIXEL_SIZE_MICRONS = 0.106
SCALE_BAR_LENGTH_MICRONS = 10

In [3]:
def add_scale_bar(
    image: np.ndarray,
    pixel_size_microns: float,
    bar_length_microns: float = 10,
    bar_thickness: int = 10,
    padding_pixels: int = 10,
):
    """
    Add a scale bar to the bottom right corner of the image.

    Parameters:
    - image: 2D numpy array representing the image
    - pixel_size_microns: size of one pixel in microns
    - bar_length_microns: desired length of the scale bar in microns (default 10 microns)
    - bar_thickness: thickness of the scale bar in pixels (default 10 pixels)

    Returns:
    - image_with_bar: copy of the input image with the scale bar added
    """
    # Calculate the length of the scale bar in pixels
    bar_length_pixels = int(bar_length_microns / pixel_size_microns)

    # Create a copy of the image to draw on
    image_with_bar = np.array(image, copy=True)

    # Define the position for the scale bar (bottom right corner with some padding)
    start_x = image.shape[1] - padding_pixels - bar_length_pixels
    start_y = image.shape[0] - padding_pixels - bar_thickness

    # Draw the scale bar as white for RGB images, or as an over-range value for grayscale images
    if image_with_bar.ndim == 2:
        image_with_bar = image_with_bar.astype(np.float32, copy=False)
        image_with_bar[
            start_y : start_y + bar_thickness, start_x : start_x + bar_length_pixels
        ] = image_with_bar.max() + 1
    else:
        image_with_bar[
            start_y : start_y + bar_thickness, start_x : start_x + bar_length_pixels
        ] = 255

    return image_with_bar


def generate_montage(
    pixel_size_microns: float = PIXEL_SIZE_MICRONS,
    bar_length_microns: float = SCALE_BAR_LENGTH_MICRONS,
    full_montage: bool = True,
    composite_only: bool = False,
    **kwargs,
) -> plt.Figure:
    """Generate a montage of the 4 channels with a scale bar added to the composite RGB image.

    Parameters
    ----------
    pixel_size_microns: float
        Size of each pixel in microns
    bar_length_microns: float
        Length of the scale bar in microns
    kwargs: dictionary containing the following keys:
        agp_image_path: pathlib.Path
            file path to the AGP channel image
        dna_image_path: pathlib.Path
            file path to the DNA channel image
        er_image_path: pathlib.Path
            file path to the ER channel image
        mito_image_path: pathlib.Path
            file path to the Mito channel image
        contrast_dict: dictionary containing contrast limits for each channel (optional, if not provided will use 99.9th percentile for each channel)


    Returns
    -------
    montage: plt.Figure
        Matplotlib figure containing the montage of the 4 channels with scale bar
    """

    if composite_only and full_montage:
        raise ValueError(
            "Cannot set both composite_only and full_montage to True. Please choose one or the other."
        )

    # load images
    agp_image = tifffile.imread(kwargs["agp_image_path"])
    dna_image = tifffile.imread(kwargs["dna_image_path"])
    er_image = tifffile.imread(kwargs["er_image_path"])
    mito_image = tifffile.imread(kwargs["mito_image_path"])
    contrast_dict = kwargs.get("contrast_dict", None)

    # define cmaps: DNA -> cyan, Mito -> magenta, AGP -> green, ER -> red
    cyan_cmap = LinearSegmentedColormap.from_list("cyan_cmap", [(0, 0, 0), (0, 1, 1)])
    magenta_cmap = LinearSegmentedColormap.from_list(
        "magenta_cmap", [(0, 0, 0), (1, 0, 1)]
    )
    green_cmap = LinearSegmentedColormap.from_list("green_cmap", [(0, 0, 0), (0, 1, 0)])
    red_cmap = LinearSegmentedColormap.from_list("red_cmap", [(0, 0, 0), (1, 0, 0)])
    # ensure any over-range pixels (scale bar) render white
    cyan_cmap.set_over("white")
    magenta_cmap.set_over("white")
    green_cmap.set_over("white")
    red_cmap.set_over("white")

    # normalize each channel for display (clip to 99.9th percentile to reduce impact of outliers, then scale to [0, 255])
    def normalize_for_display(image, percentile=99.9, min_value=0, max_value=255):
        p999 = np.percentile(image, percentile)
        image_clipped = np.clip(image, min_value, p999)
        image_normalized = (image_clipped / p999) * (max_value - min_value) + min_value
        return image_normalized.astype(np.uint8)

    if contrast_dict:
        agp_display = normalize_for_display(
            agp_image,
            percentile=99.9,
            min_value=contrast_dict.get("AGP", (0, 255))[0],
            max_value=contrast_dict.get("AGP", (0, 255))[1],
        )
        dna_display = normalize_for_display(
            dna_image,
            percentile=99.9,
            min_value=contrast_dict.get("DNA", (0, 255))[0],
            max_value=contrast_dict.get("DNA", (0, 255))[1],
        )
        er_display = normalize_for_display(
            er_image,
            percentile=99.9,
            min_value=contrast_dict.get("ER", (0, 255))[0],
            max_value=contrast_dict.get("ER", (0, 255))[1],
        )
        mito_display = normalize_for_display(
            mito_image,
            percentile=99.9,
            min_value=contrast_dict.get("Mito", (0, 255))[0],
            max_value=contrast_dict.get("Mito", (0, 255))[1],
        )
    else:
        agp_display = normalize_for_display(agp_image, percentile=99.9)
        dna_display = normalize_for_display(dna_image, percentile=99.9)
        er_display = normalize_for_display(er_image, percentile=99.9)
        mito_display = normalize_for_display(mito_image, percentile=99.9)

    # add white scale bars to the single-channel display images
    agp_display_with_scale_bar = add_scale_bar(
        agp_display,
        pixel_size_microns=PIXEL_SIZE_MICRONS,
        bar_length_microns=SCALE_BAR_LENGTH_MICRONS,
    )
    er_display_with_scale_bar = add_scale_bar(
        er_display,
        pixel_size_microns=PIXEL_SIZE_MICRONS,
        bar_length_microns=SCALE_BAR_LENGTH_MICRONS,
    )
    mito_display_with_scale_bar = add_scale_bar(
        mito_display,
        pixel_size_microns=PIXEL_SIZE_MICRONS,
        bar_length_microns=SCALE_BAR_LENGTH_MICRONS,
    )
    dna_display_with_scale_bar = add_scale_bar(
        dna_display,
        pixel_size_microns=PIXEL_SIZE_MICRONS,
        bar_length_microns=SCALE_BAR_LENGTH_MICRONS,
    )

    # now take the images from the 4 channels and combine them into a single RGB image for visualization
    # normalize each channel to the range [0, 1]
    agp_norm = agp_display.astype(np.float32) / 255
    er_norm = er_display.astype(np.float32) / 255
    mito_norm = mito_display.astype(np.float32) / 255
    dna_norm = dna_display.astype(np.float32) / 255
    # build an RGB composite mapping: AGP->green, Mito->magenta (red+blue), DNA->cyan (green+blue), ER->red (reduced)
    rgb_image = np.zeros((*agp_norm.shape, 3), dtype=np.float32)
    rgb_image[..., 0] += mito_norm * 1.0  # red from mito (magenta)
    rgb_image[..., 0] += er_norm * 0.6  # red from ER (reduced)
    rgb_image[..., 1] += agp_norm * 1.0  # green from AGP
    rgb_image[..., 1] += dna_norm * 0.8  # green from DNA (cyan)
    rgb_image[..., 2] += mito_norm * 1.0  # blue from mito (magenta)
    rgb_image[..., 2] += dna_norm * 1.0  # blue from DNA (cyan)
    # clip to valid range
    rgb_image = np.clip(rgb_image, 0, 1)
    rgb_image_with_scale_bar = add_scale_bar(
        (rgb_image * 255).astype(np.uint8),
        pixel_size_microns=PIXEL_SIZE_MICRONS,
        bar_length_microns=SCALE_BAR_LENGTH_MICRONS,
    )

    if full_montage and not composite_only:
        # show the 5 image montage with the composite image in the center and the 4 individual channels around it
        fig, axs = plt.subplots(1, 5, figsize=(20, 10))
        axs[0].imshow(agp_display_with_scale_bar, cmap=green_cmap, vmin=0, vmax=255)
        axs[0].set_title("AGP")
        axs[1].imshow(dna_display_with_scale_bar, cmap=cyan_cmap, vmin=0, vmax=255)
        axs[1].set_title("DNA")
        axs[2].imshow(er_display_with_scale_bar, cmap=red_cmap, vmin=0, vmax=255)
        axs[2].set_title("ER")
        axs[3].imshow(mito_display_with_scale_bar, cmap=magenta_cmap, vmin=0, vmax=255)
        axs[3].set_title("Mito")
        axs[4].imshow(rgb_image_with_scale_bar)
        axs[4].set_title("Composite")
        # remove axes for all subplots
        for ax in axs.flatten():
            ax.axis("off")
        plt.tight_layout()
        plt.close(fig)
        return fig
    elif composite_only and not full_montage:
        fig, ax = plt.subplots(figsize=(10, 10))
        ax.imshow(rgb_image_with_scale_bar)
        ax.axis("off")
        plt.tight_layout()
        plt.close(fig)
        return fig

## Get the paths

In [4]:
root_dir, in_notebook = init_notebook()

image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)
image_base_dir = image_base_dir / "data"

drugs_plate_map_path = pathlib.Path(
    f"{root_dir}/config/platemaps/NF0014_T1_platemap.csv"
)
drugs_df = pd.read_csv(drugs_plate_map_path)

In [5]:
output_dict = {
    "patient": [],
    "well_fov": [],
    "AGP_file_path": [],
    "DNA_file_path": [],
    "ER_file_path": [],
    "Mito_file_path": [],
    "Trans_file_path": [],
}
patients = image_base_dir.glob("*")
patients = [
    pathlib.Path(f"{p}/2D_analysis/0a.zmax_proj") for p in patients if p.is_dir()
]
for patient_path in patients:
    well_fovs = patient_path.glob("*")
    well_fovs = [f for f in well_fovs if f.is_dir()]
    for well_fov in well_fovs:
        if well_fov.name in ["run_stats"]:
            continue
        files = [x for x in well_fov.glob("*") if x.is_file()]
        for f in files:
            if "555" in f.name:
                output_dict["AGP_file_path"].append(str(f))
            elif "405" in f.name:
                output_dict["DNA_file_path"].append(str(f))
            elif "488" in f.name:
                output_dict["ER_file_path"].append(str(f))
            elif "640" in f.name:
                output_dict["Mito_file_path"].append(str(f))
            elif "TRANS" in f.name:
                output_dict["Trans_file_path"].append(str(f))

        output_dict["patient"].append(patient_path.parent.parent.name)
        output_dict["well_fov"].append(well_fov.name)
df = pd.DataFrame(output_dict)
df.insert(2, "well", df["well_fov"].apply(lambda x: x.split("-")[0]))
df = df.merge(drugs_df, left_on=["well"], right_on=["WellPosition"], how="left")
df.dropna(inplace=True)
df["Dose"] = df["Dose"].astype(int)
df["full_treatment_name"] = df["Treatment"] + " " + df["Dose"].astype(str) + df["Unit"]
df["full_treatment_name_machine_readable"] = (
    df["Treatment"] + "_" + df["Dose"].astype(str) + df["Unit"]
)
df.head()

,patient,well_fov,well,AGP_file_path,DNA_file_path,ER_file_path,Mito_file_path,Trans_file_path,WellRow,WellCol,WellPosition,Treatment,Dose,Unit,full_treatment_name,full_treatment_name_machine_readable
0,NF0014_T2,C11-4,C11,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,C,11.0,C11,Staurosporine,10,nM,Staurosporine 10nM,Staurosporine_10nM
1,NF0014_T2,D11-5,D11,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,D,11.0,D11,Selumetinib,1,uM,Selumetinib 1uM,Selumetinib_1uM
2,NF0014_T2,G2-5,G2,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,G,2.0,G2,Staurosporine,10,nM,Staurosporine 10nM,Staurosporine_10nM
3,NF0014_T2,C6-6,C6,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,C,6.0,C6,Imatinib,1,uM,Imatinib 1uM,Imatinib_1uM
4,NF0014_T2,E5-2,E5,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,E,5.0,E5,Fimepinostat,1,uM,Fimepinostat 1uM,Fimepinostat_1uM


In [6]:
dict_of_dmso_examples = {
    "patient": [
        "NF0014_T1",
        "NF0014_T2",
        "NF0016_T1",
        "NF0018_T6",
        "NF0021_T1",
        "NF0030_T1",
        "NF0035_T1",
        "NF0037_T1",
        "NF0040_T1",
        "NF0055_T1",
        "SARCO219_T2",
        "SARCO361_T1",
    ],
    "well_fov": [
        "C4-2",  # 14 T1
        "E4-4",  # 14 T2
        "D4-1",  # 16 T1
        "E9-3",  # 18 T6
        "E9-6",  # 21 T1
        "F9-3",  # 30 T1
        "G9-7",  # 35 T1
        "D9-16",  # 37 T1
        "F9-5",  # 40 T1
        "D4-7",  # 55 T1
        "E9-3",  # SARCO219 T2
        "G9-3",  # SARCO361 T1
    ],
    "agp_max": [
        255,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        125,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
    "dna_max": [
        125,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        115,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
    "er_max": [
        255,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        255,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
    "mito_max": [
        255,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        255,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
}

dict_of_selumetnib_1um_examples = {
    "patient": [
        "NF0014_T1",
        "NF0014_T2",
        "NF0016_T1",
        "NF0018_T6",
        "NF0021_T1",
        "NF0030_T1",
        "NF0035_T1",
        "NF0037_T1",
        "NF0040_T1",
        "NF0055_T1",
        "SARCO219_T2",
        "SARCO361_T1",
    ],
    "well_fov": [
        "G10-3",  # 14 T1
        "D11-2",  # 14 T2
        "D11-2",  # 16 T1
        "G10-3",  # 18 T6
        "G10-3",  # 21 T1
        "G10-1",  # 30 T1
        "G10-1",  # 35 T1
        "G10-1",  # 37 T1
        "G10-1",  # 40 T1
        "G10-1",  # 55 T1
        "G10-2",  # SARCO219 T2
        "G10-3",  # SARCO361 T1
    ],
    "agp_max": [
        155,  # 14 T1
        255,  # 14 T2
        155,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        125,  # 30 T1
        155,  # 35 T1
        155,  # 37 T1
        155,  # 40 T1
        125,  # 55 T1
        155,  # SARCO219 T2
        155,  # SARCO361 T1
    ],
    "dna_max": [
        125,  # 14 T1
        255,  # 14 T2
        155,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        115,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        155,  # 40 T1
        255,  # 55 T1
        155,  # SARCO219 T2
        155,  # SARCO361 T1
    ],
    "er_max": [
        255,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        255,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
    "mito_max": [
        155,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        55,  # 30 T1
        125,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        155,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
}

dict_of_selumetnib_10um_examples = {
    "patient": [
        "NF0014_T1",
        "NF0014_T2",
        "NF0016_T1",
        "NF0018_T6",
        "NF0021_T1",
        "NF0030_T1",
        "NF0035_T1",
        "NF0037_T1",
        "NF0040_T1",
        "NF0055_T1",
        "SARCO219_T2",
        "SARCO361_T1",
    ],
    "well_fov": [
        "E11-1",  # 14 T1
        "F11-2",  # 14 T2
        "F11-2",  # 16 T1
        "E11-3",  # 18 T6
        "F11-5",  # 21 T1
        "F11-1",  # 30 T1
        "F11-1",  # 35 T1
        "F11-2",  # 37 T1
        "F11-2",  # 40 T1
        "E11-1",  # 55 T1
        "F11-1",  # SARCO219 T2
        "F11-2",  # SARCO361 T1
    ],
    "agp_max": [
        255,  # 14 T1
        255,  # 14 T2
        155,  # 16 T1
        255,  # 18 T6
        155,  # 21 T1
        125,  # 30 T1
        155,  # 35 T1
        155,  # 37 T1
        155,  # 40 T1
        155,  # 55 T1
        155,  # SARCO219 T2
        155,  # SARCO361 T1
    ],
    "dna_max": [
        125,  # 14 T1
        255,  # 14 T2
        155,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        115,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        155,  # 40 T1
        255,  # 55 T1
        155,  # SARCO219 T2
        155,  # SARCO361 T1
    ],
    "er_max": [
        255,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        255,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
    "mito_max": [
        155,  # 14 T1
        255,  # 14 T2
        100,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        55,  # 30 T1
        155,  # 35 T1
        255,  # 37 T1
        155,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
}

dict_of_mirdametinib_1um_examples = {
    "patient": [
        "NF0014_T1",
        "NF0014_T2",
        "NF0016_T1",
        "NF0018_T6",
        "NF0021_T1",
        "NF0030_T1",
        "NF0035_T1",
        "NF0037_T1",
        "NF0040_T1",
        "NF0055_T1",
        "SARCO219_T2",
        "SARCO361_T1",
    ],
    "well_fov": [
        "E8-1",  # 14 T1
        "E8-2",  # 14 T2
        "E8-2",  # 16 T1
        "E8-1",  # 18 T6
        "F8-2",  # 21 T1
        "F8-1",  # 30 T1
        "E8-1",  # 35 T1
        "F8-2",  # 37 T1
        "F8-2",  # 40 T1
        "F8-4",  # 55 T1
        "F8-3",  # SARCO219 T2
        "F8-3",  # SARCO361 T1
    ],
    "agp_max": [
        255,  # 14 T1
        255,  # 14 T2
        175,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        125,  # 30 T1
        155,  # 35 T1
        155,  # 37 T1
        200,  # 40 T1
        150,  # 55 T1
        155,  # SARCO219 T2
        100,  # SARCO361 T1
    ],
    "dna_max": [
        125,  # 14 T1
        255,  # 14 T2
        155,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        115,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        155,  # 40 T1
        255,  # 55 T1
        155,  # SARCO219 T2
        155,  # SARCO361 T1
    ],
    "er_max": [
        255,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        255,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
    "mito_max": [
        255,  # 14 T1
        255,  # 14 T2
        125,  # 16 T1
        155,  # 18 T6
        200,  # 21 T1
        55,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        155,  # 55 T1
        155,  # SARCO219 T2
        55,  # SARCO361 T1
    ],
}

dict_of_mirdametinib_10um_examples = {
    "patient": [
        "NF0014_T1",
        "NF0014_T2",
        "NF0016_T1",
        "NF0018_T6",
        "NF0021_T1",
        "NF0030_T1",
        "NF0035_T1",
        "NF0037_T1",
        "NF0040_T1",
        "NF0055_T1",
        "SARCO219_T2",
        "SARCO361_T1",
    ],
    "well_fov": [
        "C9-2",  # 14 T1
        "G8-3",  # 14 T2
        "G8-1",  # 16 T1
        "G8-3",  # 18 T6
        "G8-1",  # 21 T1
        "G8-1",  # 30 T1
        "G8-1",  # 35 T1
        "G8-1",  # 37 T1
        "C9-5",  # 40 T1
        "G8-2",  # 55 T1
        "G8-1",  # SARCO219 T2
        "G8-1",  # SARCO361 T1
    ],
    "agp_max": [
        255,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        155,  # 21 T1
        125,  # 30 T1
        155,  # 35 T1
        155,  # 37 T1
        200,  # 40 T1
        250,  # 55 T1
        255,  # SARCO219 T2
        125,  # SARCO361 T1
    ],
    "dna_max": [
        125,  # 14 T1
        255,  # 14 T2
        155,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        115,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        155,  # 40 T1
        255,  # 55 T1
        155,  # SARCO219 T2
        155,  # SARCO361 T1
    ],
    "er_max": [
        255,  # 14 T1
        255,  # 14 T2
        255,  # 16 T1
        255,  # 18 T6
        255,  # 21 T1
        255,  # 30 T1
        255,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        255,  # 55 T1
        255,  # SARCO219 T2
        255,  # SARCO361 T1
    ],
    "mito_max": [
        255,  # 14 T1
        85,  # 14 T2
        225,  # 16 T1
        155,  # 18 T6
        200,  # 21 T1
        55,  # 30 T1
        105,  # 35 T1
        255,  # 37 T1
        255,  # 40 T1
        250,  # 55 T1
        155,  # SARCO219 T2
        125,  # SARCO361 T1
    ],
}

In [7]:
# merge the dicts
all_examples_dict = {
    "patient": dict_of_dmso_examples["patient"]
    + dict_of_selumetnib_1um_examples["patient"]
    + dict_of_selumetnib_10um_examples["patient"]
    + dict_of_mirdametinib_1um_examples["patient"]
    + dict_of_mirdametinib_10um_examples["patient"],
    "well_fov": dict_of_dmso_examples["well_fov"]
    + dict_of_selumetnib_1um_examples["well_fov"]
    + dict_of_selumetnib_10um_examples["well_fov"]
    + dict_of_mirdametinib_1um_examples["well_fov"]
    + dict_of_mirdametinib_10um_examples["well_fov"],
    "agp_max": dict_of_dmso_examples["agp_max"]
    + dict_of_selumetnib_1um_examples["agp_max"]
    + dict_of_selumetnib_10um_examples["agp_max"]
    + dict_of_mirdametinib_1um_examples["agp_max"]
    + dict_of_mirdametinib_10um_examples["agp_max"],
    "dna_max": dict_of_dmso_examples["dna_max"]
    + dict_of_selumetnib_1um_examples["dna_max"]
    + dict_of_selumetnib_10um_examples["dna_max"]
    + dict_of_mirdametinib_1um_examples["dna_max"]
    + dict_of_mirdametinib_10um_examples["dna_max"],
    "er_max": dict_of_dmso_examples["er_max"]
    + dict_of_selumetnib_1um_examples["er_max"]
    + dict_of_selumetnib_10um_examples["er_max"]
    + dict_of_mirdametinib_1um_examples["er_max"]
    + dict_of_mirdametinib_10um_examples["er_max"],
    "mito_max": dict_of_dmso_examples["mito_max"]
    + dict_of_selumetnib_1um_examples["mito_max"]
    + dict_of_selumetnib_10um_examples["mito_max"]
    + dict_of_mirdametinib_1um_examples["mito_max"]
    + dict_of_mirdametinib_10um_examples["mito_max"],
}
all_examples_df = pd.DataFrame(all_examples_dict)

In [8]:
sampled_df = df.merge(all_examples_df, on=["patient", "well_fov"], how="inner")
print(sampled_df.shape)
# sort by patient for easier visualization
sampled_df = sampled_df.sort_values(by="patient")
sampled_df.head()

(54, 20)


,patient,well_fov,well,AGP_file_path,DNA_file_path,ER_file_path,Mito_file_path,Trans_file_path,WellRow,WellCol,WellPosition,Treatment,Dose,Unit,full_treatment_name,full_treatment_name_machine_readable,agp_max,dna_max,er_max,mito_max
39,NF0014_T1,C9-2,C9,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,C,9.0,C9,Mirdametinib,10,uM,Mirdametinib 10uM,Mirdametinib_10uM,255,125,255,255
40,NF0014_T1,E11-1,E11,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,E,11.0,E11,Selumetinib,10,uM,Selumetinib 10uM,Selumetinib_10uM,255,125,255,155
41,NF0014_T1,G10-3,G10,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,G,10.0,G10,Selumetinib,1,uM,Selumetinib 1uM,Selumetinib_1uM,155,125,255,155
42,NF0014_T1,E8-1,E8,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,E,8.0,E8,Mirdametinib,1,uM,Mirdametinib 1uM,Mirdametinib_1uM,255,125,255,255
43,NF0014_T1,C4-2,C4,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,/home/lippincm/mnt/bandicoot/NF1_organoid_data...,C,4.0,C4,DMSO,1,%,DMSO 1%,DMSO_1%,255,125,255,255


In [9]:
pathlib.Path("../montages").mkdir(exist_ok=True)
# generate montages for the example well_fovs listed above with individual contrast adjustments for each channel
for idx, row in tqdm.tqdm(sampled_df.iterrows(), total=sampled_df.shape[0]):
    fig = generate_montage(
        agp_image_path=row["AGP_file_path"],
        dna_image_path=row["DNA_file_path"],
        er_image_path=row["ER_file_path"],
        mito_image_path=row["Mito_file_path"],
        composite_only=True,
        full_montage=False,
        contrast_dict={
            "AGP": (0, row["agp_max"]),
            "DNA": (0, row["dna_max"]),
            "ER": (0, row["er_max"]),
            "Mito": (0, row["mito_max"]),
        },
    )
    fig.savefig(
        f"../montages/{row['patient']}_{row['well_fov']}_{row['full_treatment_name_machine_readable']}_composite.png",
        dpi=600,
    )

100%|██████████| 54/54 [02:11<00:00,  2.44s/it]
